**Timeline**

* Нечеткий поиск продуктов в БД
* Отдельный словарь для сложных случаев
* Обработка ложных срабатываний
* Динамический порог (для коротких названий порог выше, чем для длинных)



In [ ]:
!apt-get install tesseract-ocr -y
!apt-get install tesseract-ocr-rus -y
!pip install pytesseract pandas python-telegram-bot==20.7 opencv-python-headless psycopg2-binary fuzzywuzzy python-Levenshtein
!pip install openai --upgrade -q

print("Библиотеки установлены")

In [ ]:
import os, re, asyncio
import numpy as np
import io, logging, random
from PIL import Image, ImageEnhance
import pytesseract, cv2, psycopg2
from psycopg2 import pool
from google.colab import userdata
from fuzzywuzzy import fuzz, process
from openai import OpenAI

logging.basicConfig(level=logging.ERROR)
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

OPENAI_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_KEY)

print("Клиент инициализирован")

In [ ]:
DB_HOST = "eco-scan-ecobot.e.aivencloud.com"
DB_PORT = "10408"
DB_NAME = "defaultdb"
DB_USER = "avnadmin"
DB_PASS = userdata.get('DB_PASSWORD')

db_Pool = psycopg2.pool.SimpleConnectionPool(1, 10, host=DB_HOST, port=DB_PORT, database=DB_NAME, user=DB_USER, password=DB_PASS, sslmode='require')
print("Подключено к БД")

def load():
    conn = db_Pool.getconn()
    try:
        with conn.cursor() as cursor:
            cursor.execute("""SELECT name, matched_product_id FROM lenta WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM magnit WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM okey WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM perekrestok WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM x5 WHERE name IS NOT NULL""")
            store_prods = {}
            for name, matched_id in cursor.fetchall():
                if name and len(name) > 3:
                    store_prods[name.lower()] = matched_id

            cursor.execute("""SELECT pr.id, pr.name, c.name as category, pr.carbon_footprint FROM product_reference pr JOIN categories c ON pr.category_id = c.id""")
            products = {}
            for product_id, name, category, footprint in cursor.fetchall():
                products[product_id] = {'name': name, 'category': category, 'co2': float(footprint) if footprint else 0.0}
            print(f"Загружено названий из магазинов: {len(store_prods)}")
            print(f"Загружено продуктов из справочника: {len(products)}")
            return store_prods, products
    except:
        print("Ошибка")
        return {}, {}
    finally:
        db_Pool.putconn(conn)

store_prods, prod_info = load()

In [ ]:
def preproc_img(image_bytes):
    try:
        nparr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if img is None:
            return None

        height, width = img.shape[:2]
        if width < 1500:
            scale = 2000 / width
            new_width = int(width * scale)
            new_height = int(height * scale)
            img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)
        denoised = cv2.fastNlMeansDenoising(enhanced, h=30)
        binary = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

        pil_img = Image.fromarray(binary)
        sharpen = ImageEnhance.Sharpness(pil_img)
        shrpnd = sharpen.enhance(1.5)
        return shrpnd
    except:
        print("Ошибка предобработки")
        return None

In [ ]:
async def gpt(ocr_txt):
    if not OPENAI_KEY:
        print("Ключ OpenAI API не настроен")
        return ocr_txt

    try:
        prompt = f"""Ты эксперт по исправлению OCR-ошибок в чеках. ВАЖНЫЕ ПРАВИЛА:
1. Текст чеков на РУССКОМ языке. Не переводи ничего на английский или другие языки.
2. ТОРГОВЫЕ НАИМЕНОВАНИЯ (бренды, названия продуктов) НЕЛЬЗЯ заменять или переводить.
3. Исправляй только явные опечатки (буквы, цифры).
4. НЕ удаляй пробелы и дефисы в названиях.
5. Сохраняй оригинальную структуру строк.
6. Не переводи и не переименовывай продукты.
7. Сохраняй КАЖДУЮ строку чека.
8. НЕ УДАЛЯЙ и НЕ ОБЪЕДИНЯЙ строки с продуктами.

Оригинальный OCR-текст:
{ocr_txt}

Верни ТОЛЬКО исправленный текст, сохраняя все строки, без пояснений и комментариев."""

        response = client.chat.completions.create(model="gpt-4.1-mini", messages=[{"role": "user", "content": prompt}], max_tokens=2000, temperature=0.1)

        fxd_txt = response.choices[0].message.content
        return fxd_txt
    except:
        print("Ошибка")
        return ocr_txt

In [ ]:
async def img_txt(photo_file):
    try:
        file = await photo_file.get_file()
        ph_bytes = await file.download_as_bytearray()
        processed = preproc_img(ph_bytes)
        if processed:
            text1 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 6')
            text2 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 3')
            text3 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 11')
            combined = text1 + "\n" + text2 + "\n" + text3

            combined = await gpt(combined)

            return combined.lower()
        return ""
    except:
        print("Ошибка")
        return ""

In [ ]:
PROBLEMS = {'gal.пастила h.s.p.мед.обл.': ['салатила', 'салат пастила', 'пастила н.', 's.p.мед', 's.p.нед', 'н.5.р.нец', 'h.s.p.мед'],
    'nina farina тараллини с чесноком': ['tanai', 'farina', 'sf j)nina', 'taran', 'taranfvh'],
    'святой источник вода питьевая лимон': ['свят ион', 'свято', 'святой'],
    'хлебцы молод.лайт с вино.кос': ['executor', 'honoo', 'aaat', 'кгс7оф'],
    'салат фрилис': ['фрилис', 'phankc', 'opnankc', 'фрилик', 'фрилиc', 'салат фрилис'],
    'хлебец зож': ['xnebeu', 'зож', 'body-эф', 'body', 'хлебец зож'],
    'mentos жев.рез.p.fr.з.м.15,5г' : ['mentos', 'heb.pe3.p.fr.3.m.15'],
    'gl.vil.нект.м/фр.из см.фруко': ['фруко', 'см.фруко'],
    'джин gletcher 40% 0.25л' : ['gletcher', "'gletcher"],
    'ml.s.сыр серб.брынза мяг.': ['брынза', 'серб.брынза', 'ml.s.сыр', 'сыр серб', 'брынза мяг', 'аззне'],
    'lor.чипсы nat.мор.сол/перец.' : ['lor.чипсы', 'nat.мор.'],
    'пиво варим сусло св.нефил.' : ['варим сусло', 'baphn cycno', 'cb.heohn'],
    'ром выдержанный steersman' : ['steersman', 'ром'],}

In [ ]:
def fake(product_name, receipt_line, match_score=100):
    product_lower = product_name.lower()
    receipt_lower = receipt_line.lower()

    if 'томат' in product_lower or 'помидор' in product_lower:
        if 'томат' not in receipt_lower and 'помидор' not in receipt_lower and 'черри' not in receipt_lower:
            print(f"Отфильтрован 'Томаты' (в строке нет 'томат'/'черри')")
            return True

    if match_score < 55:
        return True

    if 'святой' in product_lower and match_score < 75:
        print(f"Отфильтрован 'Святой источник' (скор: {match_score}%)")
        return True

    return False

In [ ]:
def find_prods(text):
    found = []
    found_prods = set()

    print("\nПоиск продуктов")

    for product_name, aliases in PROBLEMS.items():
        for alias in aliases:
            if alias in text:
                product_id = store_prods.get(product_name)
                if product_id and product_id in prod_info:
                    if fake(product_name, alias, 100):
                        continue
                    product = prod_info[product_id]
                    if product_name not in found_prods:
                        found.append({'receipt_name': product_name, 'category': product['category'], 'co2': product['co2']})
                        found_prods.add(product_name)
                        print(f"Найдено по алиасу: {product_name[:40]}")
                    break

    lines = text.split('\n')
    candidates = []
    for line in lines:
        line = line.strip()
        if not line or len(line) < 4:
            continue
        clean = re.sub(r'\d+', '', line)
        clean = re.sub(r'[^\w\s]', ' ', clean)
        clean = re.sub(r'\s+', ' ', clean).strip().lower()
        if len(clean) > 3:
            candidates.append(clean)

    all_names = list(store_prods.keys())

    for c in candidates:
        already_f = any(f['receipt_name'] in c for f in found)
        if already_f:
            continue

        if len(c) < 5:
            continue

        if not re.search(r'[а-яё]', c):
            continue

        cand_len = len(c)
        if cand_len < 10:
            dynamic_thr = 80
        elif cand_len > 30:
            dynamic_thr = 55
        else:
            dynamic_thr = 65

        mtchs = process.extract(c, all_names, limit=1, scorer=fuzz.ratio)

        for mtch, score in mtchs:
            if score >= dynamic_thr:
                if fake(mtch, c, score):
                    print(f"Отфильтровао: {mtch} ({score}%)")
                    continue

                if 'банан' in mtch.lower():
                    pastila = any('пастила' in f['receipt_name'].lower() for f in found)
                    if pastila or 'салатила' in c or 'листила' in c:
                        print(f"Блокировка: {mtch}")
                        continue

                if 'томат' in mtch.lower():
                    txt_low = text.lower()
                    if 'томат' not in txt_low and 'черри' not in txt_low and 'помидор' not in txt_low:
                        print(f"Блокировка ложных томатов: {mtch} (в тексте нет признаков томатов)")
                        continue

                if 'магнит' in mtch.lower():
                    keywrds = ['сок', 'ролл', 'напиток', 'энергет', 'пиво', 'шоколад', 'сыр', 'пастила', 'хлеб']
                    prod_keyw = any(keyword in c for keyword in keywrds)
                    if not prod_keyw:
                        print(f"Блокировка ложного продукта 'Магнит': {mtch} (нет ключевых слов продукта)")
                        continue

                product_id = store_prods.get(mtch)
                if product_id and product_id in prod_info:
                    if mtch in found_prods:
                        print(f"Пропуск точного дубля: {mtch}")
                        continue
                    product = prod_info[product_id]
                    found.append({'receipt_name': mtch, 'category': product['category'], 'co2': product['co2']})
                    found_prods.add(mtch)
                    print(f"Найдено ({score}%): {mtch[:40]}")
                    break

    return found

https://pypi.org/project/pytesseract/

https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html

https://pillow.readthedocs.io/en/stable/reference/ImageEnhance.html

https://platform.openai.com/docs/api-reference

https://github.com/seatgeek/fuzzywuzzy

https://github.com/ztane/python-Levenshtein/

https://docs.python.org/3/library/asyncio.html